# Selected-Area Electron Diffraction: A Flat Slice Of Reciprocal Space

A powder pattern is a set of rings, because every crystallite orientation is present. A single-crystal
electron diffraction pattern is a set of *spots* arranged on a lattice, and the reason it looks like a
lattice is a numerical accident of electron wavelengths.

At 200 kV the electron wavelength is $0.02508$ Å, so the Ewald sphere has radius
$1/\lambda = 39.9$ Å$^{-1}$. The reciprocal-lattice vectors of a metal are around $0.5$ Å$^{-1}$. The
sphere is therefore **eighty times larger than the region it is cutting**, and over that region it is
almost a plane. A zone-axis pattern is a planar section of the reciprocal lattice, undistorted, which
is why you can measure interplanar angles off it with a ruler.

"Almost" is the interesting word. Quantify the departure and you get the excitation error, the relrod,
and the higher-order Laue zones — and the fact that spots which the structure factor forbids appear
anyway.

## Learning goals

1. How flat is the Ewald sphere at electron energies, in numbers?
2. What does the camera constant do, and why can a pattern be indexed without knowing it?
3. Why does the zone law decide which spots are present, and the structure factor decide which are
   visible?
4. What is the excitation error, and why do spots away from the zone axis fade rather than vanish?
5. Which spots in a real pattern are *not* explained by the kinematic model at all?

## 0. Setup

Nickel and zirconium, down their principal zone axes, at 200 kV.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    ReferenceFrame,
    ZoneAxis,
    generate_saed_pattern,
    get_phase_fixture,
    plot_saed_pattern,
)
from pytex.diffraction.kinematic import electron_wavelength_angstrom

np.set_printoptions(precision=4, suppress=True)

CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
NICKEL = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)
ZIRCONIUM = get_phase_fixture("zr_hcp").load_phase(crystal_frame=CRYSTAL)
BEAM_KEV = 200.0
CAMERA_CONSTANT = 180.0  # mm angstrom


def label(indices):
    return str(tuple(int(value) for value in np.asarray(indices).ravel()))


print(f"{'beam (kV)':>10} {'wavelength (A)':>16} {'Ewald radius 1/lambda (1/A)':>29}")
for energy in (100.0, 200.0, 300.0, 1000.0):
    wavelength = electron_wavelength_angstrom(energy)
    print(f"{energy:>10.0f} {wavelength:>16.5f} {1.0 / wavelength:>29.2f}")

## 1. How flat is the Ewald sphere?

The Ewald construction says a reflection $\mathbf{g}$ diffracts when $\mathbf{g}$ lies on a sphere of
radius $1/\lambda$ through the origin, centred at $-\mathbf{k}_0$. Put the incident beam along the
zone axis and ask how far a reciprocal-lattice point *in the zero-layer plane* is from that sphere.
The answer is the sagitta,

$$s_g \simeq -\frac{\lambda |\mathbf{g}|^{2}}{2},$$

so the sphere's departure from the tangent plane grows quadratically with $|\mathbf{g}|$ and linearly
with $\lambda$. For X-rays, where $\lambda \approx 1.5$ Å and $1/\lambda \approx 0.65$ Å$^{-1}$, the
sphere is *comparable* to the reciprocal lattice and there is no flat slice at all — which is exactly
why single-crystal X-ray work rotates the crystal instead of reading a planar section.

In [ ]:
magnitudes = np.array([0.2, 0.5, 1.0, 2.0, 4.0])
print(f"{'|g| (1/A)':>10} {'d (A)':>8} {'electron sagitta (1/A)':>24} {'as a fraction of |g|':>22}")
wavelength = electron_wavelength_angstrom(BEAM_KEV)
for magnitude in magnitudes:
    sagitta = wavelength * magnitude**2 / 2.0
    print(f"{magnitude:>10.2f} {1.0 / magnitude:>8.3f} {sagitta:>24.5f} "
          f"{sagitta / magnitude:>22.5f}")

print(f"\nat {BEAM_KEV:.0f} kV, 1/lambda = {1.0 / wavelength:.2f} 1/A")
print(f"a typical metal reflection has |g| ~ 0.5 1/A, which is "
      f"{(1.0 / wavelength) / 0.5:.0f}x smaller than the Ewald radius")
xray_wavelength = 1.5406
print(f"\nfor Cu K-alpha, 1/lambda = {1.0 / xray_wavelength:.3f} 1/A -- comparable with |g| itself,")
print(f"and the sagitta at |g| = 0.5 is {xray_wavelength * 0.25 / 2:.4f} 1/A, "
      f"{xray_wavelength * 0.25 / 2 / 0.5 * 100:.0f} percent of |g|.")
print("There is no flat slice in the X-ray case, which is why the two techniques look")
print("nothing like each other.")

> **The awe note.** The whole practice of electron diffraction — reading interplanar angles straight
> off a photograph, indexing a pattern by inspection, recognizing a zone axis by its symmetry —
> depends on a ratio of about eighty between the Ewald radius and the reciprocal-lattice spacing, and
> that ratio is a consequence of the electron's mass. A 200 kV electron has $\lambda = 0.025$ Å because
> $\lambda = h/p$ and its momentum is large; a photon of the same energy has $\lambda = 0.062$ Å and
> would do nearly as well, but a *usable* photon source at that energy cannot be focused into a
> nanometre probe. The flatness is what makes the pattern a picture of a lattice plane rather than a
> curved section that has to be inverted — and the residual curvature, $\lambda|\mathbf{g}|^2/2$, is
> not a nuisance either: it is the only thing in a zone-axis pattern that knows about the lattice
> repeat *along* the beam, which is what tutorial 28 measures with HOLZ rings.

## 2. The zone law selects, the structure factor illuminates

Two independent conditions decide what appears on the screen.

**Geometry.** A reflection lies in the zero-order Laue zone of the zone axis $[uvw]$ when
$hu + kv + lw = 0$. That is the zone law of tutorial 17, and it is what makes the pattern a
two-dimensional lattice: the reflections form the *plane* of reciprocal space perpendicular to the
beam.

**Structure.** Whether that spot has any intensity is $|F_{hkl}|^2$, from tutorial 04.

The two are unrelated, and PyTex reports both, which means the pattern object contains spots that are
geometrically present and structurally extinct. That is deliberate — and section 6 is what happens if
you forget it.

In [ ]:
PATTERNS = {
    "Ni [001]": generate_saed_pattern(
        NICKEL, ZoneAxis(np.array([0, 0, 1]), phase=NICKEL),
        camera_constant_mm_angstrom=CAMERA_CONSTANT, max_index=3,
    ),
    "Ni [011]": generate_saed_pattern(
        NICKEL, ZoneAxis(np.array([0, 1, 1]), phase=NICKEL),
        camera_constant_mm_angstrom=CAMERA_CONSTANT, max_index=3,
    ),
    "Ni [111]": generate_saed_pattern(
        NICKEL, ZoneAxis(np.array([1, 1, 1]), phase=NICKEL),
        camera_constant_mm_angstrom=CAMERA_CONSTANT, max_index=3,
    ),
    "Zr [0001]": generate_saed_pattern(
        ZIRCONIUM, ZoneAxis(np.array([0, 0, 1]), phase=ZIRCONIUM),
        camera_constant_mm_angstrom=CAMERA_CONSTANT, max_index=3,
    ),
}

print(f"{'pattern':<12} {'spots listed':>13} {'zone law holds':>15} {'|F| > 0':>9} "
      f"{'extinct but listed':>19}")
for name, pattern in PATTERNS.items():
    axis = np.asarray(pattern.zone_axis.indices, dtype=np.int64)
    indices = np.asarray([spot.miller_indices for spot in pattern.spots], dtype=np.int64)
    intensities = np.asarray([spot.intensity for spot in pattern.spots])
    zone_law = np.all(indices @ axis == 0)
    visible = intensities > 1e-6 * max(intensities.max(), 1e-30)
    print(f"{name:<12} {len(pattern.spots):>13} {str(bool(zone_law)):>15} "
          f"{int(visible.sum()):>9} {int((~visible).sum()):>19}")
    assert zone_law, name

Every listed spot satisfies the zone law exactly — integer arithmetic, no tolerance — and in the
face-centred patterns most of them carry no intensity at all. Note the last row: the hexagonal
$[0001]$ pattern has *no* extinct spots, because the hcp condition ($h+2k = 3n$ with $l$ odd)
cannot fire when every reflection in the zone has $l = 0$. The extinction pattern is a property of
the zone as much as of the structure.

Look at the two conditions separately for the nickel $[001]$ pattern.

In [ ]:
pattern = PATTERNS["Ni [001]"]
indices = np.asarray([spot.miller_indices for spot in pattern.spots], dtype=np.int64)
intensities = np.asarray([spot.intensity for spot in pattern.spots])
radii = np.linalg.norm(np.asarray([spot.detector_coordinates for spot in pattern.spots]), axis=1)
magnitudes = np.linalg.norm(
    np.asarray([spot.reciprocal_vector_crystal for spot in pattern.spots]), axis=1
)

order = np.argsort(radii)
print(f"{'hkl':<10} {'|g| (1/A)':>10} {'d (A)':>8} {'R (mm)':>9} {'intensity':>13} {'visible?':>9}")
for position in order[:14]:
    print(f"{label(indices[position]):<10} {magnitudes[position]:>10.4f} "
          f"{1.0 / magnitudes[position]:>8.4f} {radii[position]:>9.3f} "
          f"{intensities[position]:>13.4g} "
          f"{str(bool(intensities[position] > 1e-6 * intensities.max())):>9}")
print("\nThe (110)-type and (100)-type spots are geometrically in the zone and structurally")
print("extinct in an fcc lattice -- so they are listed at zero intensity rather than dropped.")

## 3. The camera constant, and why a pattern indexes without it

The detector radius of a spot is

$$R = L\lambda\,|\mathbf{g}| \;\equiv\; C\,|\mathbf{g}| = \frac{C}{d},$$

with $C = L\lambda$ the **camera constant**, in mm·Å. It has to be calibrated — the effective camera
length depends on the lens settings — and that calibration is the weakest number in any quantitative
SAED measurement.

But it cancels in a *ratio*. For two spots,

$$\frac{R_1}{R_2} = \frac{|\mathbf{g}_1|}{|\mathbf{g}_2|} = \frac{d_2}{d_1},$$

with no $C$ and no $\lambda$. So a pattern can be indexed from the ratios of its spot distances and
the angles between them, and the camera constant is needed only to convert to absolute $d$-spacings.
That is why hand indexing works from an uncalibrated photograph.

In [ ]:
visible_mask = intensities > 1e-6 * intensities.max()
visible_radii = radii[visible_mask]
visible_magnitudes = magnitudes[visible_mask]
visible_indices = indices[visible_mask]

print(f"R = C |g| with C = {CAMERA_CONSTANT:.1f} mm A:")
print(f"  largest deviation of R / |g| from C: "
      f"{np.abs(visible_radii / visible_magnitudes - CAMERA_CONSTANT).max():.3e} mm A")
assert np.allclose(visible_radii / visible_magnitudes, CAMERA_CONSTANT)

# The same pattern at two camera lengths: the ratios do not move. Index 4 so that
# three rings are present.
def visible_rings(constant):
    current = generate_saed_pattern(
        NICKEL, ZoneAxis(np.array([0, 0, 1]), phase=NICKEL),
        camera_constant_mm_angstrom=constant, max_index=4,
    )
    values = np.asarray([spot.intensity for spot in current.spots])
    keep = values > 1e-6 * values.max()
    lengths = np.linalg.norm(
        np.asarray([spot.detector_coordinates for spot in current.spots])[keep], axis=1
    )
    return np.unique(np.round(lengths[lengths > 1e-6], 4))


print(f"\n{'C (mm A)':>10} {'first three ring radii (mm)':>30} {'R2/R1':>9} {'R3/R1':>9}")
ratios = []
for constant in (CAMERA_CONSTANT, 3.0 * CAMERA_CONSTANT):
    rings = visible_rings(constant)
    ratios.append((rings[1] / rings[0], rings[2] / rings[0]))
    print(f"{constant:>10.1f} {str(np.round(rings[:3], 2)):>30} "
          f"{rings[1] / rings[0]:>9.5f} {rings[2] / rings[0]:>9.5f}")
print(f"\nThe absolute radii triple; the ratios are identical to "
      f"{max(abs(ratios[0][0] - ratios[1][0]), abs(ratios[0][1] - ratios[1][1])):.1e}.")
print(f"They are sqrt(2) = {np.sqrt(2.0):.5f} and 2 for the (200), (220), (400) sequence of an")
print("fcc [001] pattern -- a fingerprint that needs no calibration at all.")
assert np.allclose(ratios[0], ratios[1])

## 4. The pattern is the reciprocal lattice plane, so its symmetry is the zone's

Because the section is planar and undistorted, the arrangement of spots *is* the two-dimensional
lattice of the zero-order Laue zone. Different zone axes therefore give visibly different symmetry —
fourfold down $[001]$, sixfold down $[111]$ of a cubic crystal, rectangular down $[011]$ — and that is
how a zone axis is recognized by eye.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15.0, 4.1))
for ax, (name, pattern) in zip(axes, PATTERNS.items()):
    plot_saed_pattern(pattern, ax=ax)
    ax.set_title(name, fontsize=10)
fig.tight_layout()

In [ ]:
print(f"{'pattern':<12} {'visible spots':>14} {'closest ring R (mm)':>21} {'d (A)':>8} "
      f"{'ring multiplicity':>18}")
for name, current in PATTERNS.items():
    spot_intensities = np.asarray([spot.intensity for spot in current.spots])
    keep = spot_intensities > 1e-6 * spot_intensities.max()
    spot_radii = np.linalg.norm(
        np.asarray([spot.detector_coordinates for spot in current.spots])[keep], axis=1
    )
    positive = spot_radii[spot_radii > 1e-6]
    smallest = positive.min()
    multiplicity = int(np.sum(np.abs(positive - smallest) < 1e-4))
    print(f"{name:<12} {int(keep.sum()):>14} {smallest:>21.3f} "
          f"{CAMERA_CONSTANT / smallest:>8.4f} {multiplicity:>18}")
print("\nThe multiplicity of the innermost ring is the rotational symmetry of the pattern:")
print("4 down cubic [001], 6 down cubic [111] and hexagonal [0001], 2 down cubic [011].")
print("That count is what an operator uses to recognize a zone axis on the screen.")

## 5. The excitation error: spots fade, they do not switch off

Away from exact zone-axis alignment a reflection is not simply on or off. Its amplitude depends on how
far it sits from the Ewald sphere, and because the crystal is a *thin foil* the reciprocal-lattice
points are not points: they are rods elongated along the beam, of length $\sim 2/t$ for thickness $t$.
That is the **relrod**, and it is why a slightly misaligned reflection still contributes.

The kinematic amplitude for excitation error $s$ and thickness $t$ is

$$A \propto \frac{\sin(\pi t s)}{\pi s},$$

which is $t$ at $s = 0$ and falls off with the first zero at $s = 1/t$. `SAEDSpot` carries
`excitation_error_inv_angstrom`, so the relrod is available even though this surface does not weight
the intensities by it.

In [ ]:
errors = np.asarray([spot.excitation_error_inv_angstrom for spot in pattern.spots])
print(f"excitation errors on the exact [001] zone axis: max |s| = {np.abs(errors).max():.3e} 1/A")
print("Zero to round-off, because every spot is in the zero-order Laue zone by construction.\n")

thicknesses = (200.0, 500.0, 1000.0)
excitation = np.linspace(-0.03, 0.03, 1200)
fig, ax = plt.subplots(figsize=(7.6, 4.2))
for thickness in thicknesses:
    amplitude = np.where(
        np.abs(excitation) < 1e-12,
        thickness,
        np.sin(np.pi * thickness * excitation) / (np.pi * excitation + 1e-300),
    )
    ax.plot(excitation, (amplitude / thickness) ** 2, lw=1.2,
            label=f"t = {thickness:.0f} A, first zero at {1.0 / thickness:.4f} 1/A")
ax.set_xlabel(r"excitation error $s$ (Å$^{-1}$)")
ax.set_ylabel(r"relative intensity $\left[\sin(\pi t s)/(\pi t s)\right]^2$")
ax.set_title("the relrod: a thinner foil tolerates more misalignment")
ax.legend(fontsize=8)
fig.tight_layout()

print(f"{'thickness (A)':>14} {'first zero s (1/A)':>19} {'tilt that reaches it (deg)':>28}")
for thickness in thicknesses:
    # s = -lambda g^2 / 2 for a zero-layer reflection tilted by theta: ds/dtheta = g.
    tilt = np.degrees((1.0 / thickness) / 0.5)
    print(f"{thickness:>14.0f} {1.0 / thickness:>19.5f} {tilt:>28.3f}")
print("\nFor a 500 A foil and a typical g of 0.5 1/A, a tilt of about a quarter of a degree")
print("moves a reflection from its maximum to its first zero. That is the sensitivity that")
print("makes two-beam imaging possible, and it is why a zone axis has to be found rather")
print("than assumed.")

## 6. Failure modes, deliberately triggered

**(a) Reading the spot list as the visible pattern.** The list is everything the zone law admits, and
for a face-centred structure *most* of it is structurally extinct — 40 of the 48 spots in the nickel
$[001]$ list. Plotting it without an intensity threshold draws a far denser lattice than any
microscope shows, and the denser lattice has the wrong symmetry as well as the wrong spacing.

In [ ]:
print(f"{'pattern':<12} {'listed':>7} {'visible':>8} {'extinct':>8} {'extinct fraction':>17}")
for name, current in PATTERNS.items():
    spot_intensities = np.asarray([spot.intensity for spot in current.spots])
    visible_count = int(np.sum(spot_intensities > 1e-6 * spot_intensities.max()))
    print(f"{name:<12} {len(current.spots):>7} {visible_count:>8} "
          f"{len(current.spots) - visible_count:>8} "
          f"{1.0 - visible_count / len(current.spots):>17.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.6))
coordinates = np.asarray([spot.detector_coordinates for spot in pattern.spots])
axes[0].scatter(coordinates[:, 0], coordinates[:, 1], s=22, color="#d62728")
axes[0].set_title("every listed spot (wrong)")
keep = intensities > 1e-6 * intensities.max()
sizes = 8.0 + 60.0 * (intensities[keep] / intensities.max()) ** 0.4
axes[1].scatter(coordinates[keep, 0], coordinates[keep, 1], s=sizes, color="#1f77b4")
axes[1].set_title("weighted by intensity (right)")
for ax in axes:
    ax.set_aspect("equal"), ax.set_xlabel("detector x (mm)"), ax.set_ylabel("detector y (mm)")
fig.tight_layout()

**(b) Assuming a forbidden spot can always come back by double diffraction.** A beam already
diffracted by one plane acts as the incident beam for another, so $\mathbf{g}_1 + \mathbf{g}_2$ can
appear even when $F(\mathbf{g}_1 + \mathbf{g}_2) = 0$. That is **double diffraction**, and it is the
standard explanation for forbidden spots in real patterns.

It cannot happen in every structure, and which structures it can happen in is decidable. Double
diffraction needs two *allowed* reflections whose sum is the forbidden one, so the question is whether
the allowed set is closed under addition — and the answer separates the two kinds of systematic
absence.

In [ ]:
def geometric_factor(phase, hkl):
    sites = phase.unit_cell.sites
    fractional = np.asarray([site.fractional_coordinates for site in sites], dtype=np.float64)
    occupancy = np.asarray([site.occupancy for site in sites], dtype=np.float64)
    return np.abs(np.sum(occupancy * np.exp(2j * np.pi * (np.asarray(hkl, dtype=np.float64)
                                                          @ fractional.T)), axis=-1))


grid = np.stack(np.meshgrid(*[np.arange(-4, 5)] * 3, indexing="ij"), axis=-1).reshape(-1, 3)
grid = grid[np.any(grid != 0, axis=1)]

print(f"{'structure':<10} {'absence from':<18} {'allowed':>8} {'forbidden':>10} "
      f"{'revivable by g1+g2':>19}")
for fixture_id, origin in (
    ("ni_fcc", "lattice centring"),
    ("fe_bcc", "lattice centring"),
    ("diamond", "centring + glide"),
    ("zr_hcp", "screw axis"),
):
    phase = get_phase_fixture(fixture_id).load_phase(crystal_frame=CRYSTAL)
    magnitude = geometric_factor(phase, grid)
    allowed = grid[magnitude > 1e-9]
    forbidden = grid[magnitude <= 1e-9]
    sums = (allowed[:, None, :] + allowed[None, :, :]).reshape(-1, 3)
    reachable = {tuple(int(value) for value in row) for row in sums}
    revivable = [row for row in forbidden if tuple(int(value) for value in row) in reachable]
    print(f"{fixture_id:<10} {origin:<18} {len(allowed):>8} {len(forbidden):>10} "
          f"{len(revivable):>19}")
    if fixture_id == "diamond":
        assert (2, 2, 2) in {tuple(int(v) for v in row) for row in revivable}
        print(f"           diamond (222) is revivable: (111) + (111), both allowed")
    if fixture_id == "ni_fcc":
        assert not revivable

> **The second awe note.** Zero for nickel and iron, and *every* forbidden reflection for
> zirconium. The reason is structural. A centring absence is a statement that the allowed
> reflections form a **sublattice** of the reciprocal lattice — all-even for fcc, $h+k+l$ even for
> bcc — and a sublattice is closed under addition, so no sum of two allowed vectors can ever land on
> a forbidden one. Double diffraction is powerless against a centring absence.
>
> A screw-axis or glide-plane absence is different: it comes from the *motif*, not the lattice, and
> the allowed set is not closed. Hence hcp $(0001)$ — famously visible in real zirconium and titanium
> patterns — and the diamond $(222)$, reachable as $(111) + (111)$ with both terms strong. So the
> answer to "will this forbidden spot show up?" is not a matter of degree at all; it is decided by
> where the absence came from, and it is decidable by integer arithmetic on the allowed set.

**(c) Treating a single pattern as an orientation.** A zone-axis pattern determines the beam direction
and the in-plane rotation, but $[uvw]$ and $[\bar u\bar v\bar w]$ give the *same* pattern, and so do all
the symmetry-equivalent axes. One pattern therefore leaves a finite ambiguity, which tutorial 27
resolves by using two.

In [ ]:
up = generate_saed_pattern(NICKEL, ZoneAxis(np.array([0, 1, 1]), phase=NICKEL),
                           camera_constant_mm_angstrom=CAMERA_CONSTANT, max_index=2)
down = generate_saed_pattern(NICKEL, ZoneAxis(np.array([0, -1, -1]), phase=NICKEL),
                             camera_constant_mm_angstrom=CAMERA_CONSTANT, max_index=2)


def radial_signature(current):
    values = np.asarray([spot.intensity for spot in current.spots])
    keep = values > 1e-6 * values.max()
    lengths = np.linalg.norm(
        np.asarray([spot.detector_coordinates for spot in current.spots])[keep], axis=1
    )
    return np.sort(np.round(lengths, 6))


print(f"[011] and [0-1-1] patterns:")
print(f"  same number of visible spots      : "
      f"{len(radial_signature(up)) == len(radial_signature(down))}")
print(f"  identical sorted radii            : "
      f"{np.allclose(radial_signature(up), radial_signature(down))}")
print("\nThe two are indistinguishable, because reversing the beam direction reverses every")
print("g and the pattern is centrosymmetric by Friedel's law. A single pattern cannot tell")
print("you which way the crystal faces.")
assert np.allclose(radial_signature(up), radial_signature(down))

## 7. What this implementation does not do

- **Single scattering only.** No double diffraction, no dynamical intensities, no thickness
  dependence. The extinct spots of section 6(b) are reported as exactly zero, which is the
  kinematic truth and not what a microscope shows. Tutorials 28 and 29 take up the dynamical
  treatment.
- **Zero-order Laue zone only.** The pattern is the exact planar section, so the higher-order Laue
  zones — the rings whose radii carry the lattice repeat along the beam — are absent. That is what
  the CBED surface adds.
- **The excitation error is carried, not applied.** `excitation_error_inv_angstrom` is on every spot,
  but intensities are not weighted by the relrod, so this is an exactly-aligned pattern rather than a
  tilted one.
- **Intensities use the atomic-number electron proxy.** $f_e(s)$ is replaced by $Z$ in the default
  model, so relative spot intensities within a ring are right and the fall-off with $|\mathbf{g}|$ is
  not. `pytex.diffraction.scattering` has the Mott–Bethe path for absolute work.
- **No instrument.** No lens distortion, no camera-length calibration model, no detector point-spread
  function; the camera constant is a number the caller supplies.

## 8. What to take away

- **The Ewald sphere is eighty times bigger than the lattice it cuts.** That ratio is why an electron
  zone-axis pattern is a flat, undistorted section of reciprocal space and an X-ray pattern is not.
- **Zone law selects, structure factor illuminates.** Both conditions are in the spot list, and they
  are independent: a third of the listed spots are geometrically present and structurally extinct.
- **Ratios index a pattern; the camera constant only scales it.** $R_1/R_2 = d_2/d_1$ with no $C$ and
  no $\lambda$, verified across a threefold change of camera length.
- **The innermost ring's multiplicity is the pattern's symmetry.** Four down cubic $[001]$, six down
  $[111]$ and hexagonal $[0001]$, two down $[011]$.
- **A quarter of a degree matters.** For a 500 Å foil the relrod's first zero is a tilt of about
  $0.23^\circ$ away, which is why a zone axis is found rather than assumed.
- **Whether a forbidden spot can come back is decided by where its absence came from.** Centring
  absences make the allowed set a sublattice, closed under addition, so double diffraction cannot
  revive them — zero of nickel's 540 and zero of iron's 364. Screw and glide absences are not closed,
  and every one of zirconium's 108 is revivable.

### Further reading

- Tutorial 04, *Phases, lattices, space groups and CIF* — the structure factors that decide which
  spots are visible.
- Tutorial 17, *Miller vectorized workflows* — the zone law that decides which are present.
- Tutorial 27, *TEM diffraction pattern indexing as a round trip* — resolving the ambiguity of
  section 6(c) with a second pattern.
- Tutorial 28, *Convergent-beam electron diffraction* — the HOLZ rings that measure what a flat
  section cannot see.
- Tutorial 30, *Kikuchi maps* — getting to the zone axis in the first place.
- `docs/tex/algorithms/reciprocal_space_and_kinematic_spots.tex` — the Ewald construction, the
  excitation error, and the camera-constant geometry.
- D. B. Williams and C. B. Carter, *Transmission Electron Microscopy*, 2nd ed. (Springer, 2009),
  Ch. 11–12 and 16 — the Ewald sphere at electron energies, relrods, and double diffraction.
- J. W. Edington, *Practical Electron Microscopy in Materials Science*, Monograph 2 (Philips, 1975) —
  indexing patterns from distance ratios without calibration.